# Lecture 21 - Testing and code review

**Lecture Topic**: How to check code you did not write. The kinds of test that catch different failures, validating the data itself, and reviewing an agent's pull request.

## Announcements

- **Mon Nov 23 includes a short quiz** (about 20 minutes), a cumulative review in lieu of a final. Then the clinic on Wednesday.

- ClassBuzz poll for today: `X7CQ`

## Where we are

Module D so far: the shape of the system (architecture), then the shape of the code (structure). Today is how you gain confidence it is correct, which matters more than usual when an agent wrote most of it.

The theme: you verify an agent's output by making its mistakes *visible*. A test is a mistake turned into a failing check, and that is the Module B idea of giving the agent a verifier, now done properly.

Today:
 - Testing what you did not write (pytest)
 - Property-based tests and fuzzing
 - Golden and scaling tests
 - Validating the data itself
 - Reviewing an agent's pull request, and its tests

# Test the behaviour, not the lines

The agent wrote the implementation, and it may rewrite it tomorrow. So test what the code *promises*, its interface and behaviour, rather than how it happens to work inside. A good test still passes after a rewrite that keeps the behaviour, and fails the moment the behaviour changes.

### Question 1

An agent opens a pull request that says "refactored for speed, all tests pass." What should you check **first**?

 - A. The variable names
 - B. Whether it changed, deleted, or weakened any tests
 - C. How long CI took
 - D. The commit message

**B.** "All tests pass" means nothing if the agent also edited the tests. The first thing to read in a green pull request is the diff to the tests themselves. Only then do the passing tests mean what you think.

# pytest: a test is a function that asserts

**pytest** is the standard Python test runner. Tests live in files named `test_*.py`, and a test is just a function named `test_...` that asserts something. You run `pytest` in the terminal, it finds every test, runs it, and reports which ones failed.

To run these yourself: `pip install pytest hypothesis pytest-benchmark`. In the notebook we write the files with `%%writefile` and run pytest with `!`, exactly as you would in a terminal.

In [ ]:
# set-up for the demos: a fresh folder for the files, and make sure `!pytest` uses this environment
import os, shutil, sys
from pathlib import Path
shutil.rmtree("testing_demo", ignore_errors=True)
Path("testing_demo").mkdir()
os.environ["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ["PATH"]

Here is the code under test, a small function in `tips.py`:

In [ ]:
%%writefile testing_demo/tips.py
def tip_rate(fare, tip):
    """Tip as a fraction of the fare."""
    if fare <= 0:
        raise ValueError(f"fare must be positive, got {fare}")
    return tip / fare

And the tests, in `test_tips.py`:

In [ ]:
%%writefile testing_demo/test_tips.py
import pytest
from tips import tip_rate

def test_basic():
    assert tip_rate(10, 2) == 0.2

def test_zero_tip():
    assert tip_rate(10, 0) == 0.0

def test_thirds():
    assert tip_rate(3, 1) == pytest.approx(0.3333, abs=1e-4)    # floats: compare approximately

@pytest.mark.parametrize("fare, tip, expected", [(20, 4, 0.2), (8, 2, 0.25), (50, 0, 0.0)])
def test_many(fare, tip, expected):                            # one test, run three times
    assert tip_rate(fare, tip) == expected

def test_rejects_zero_fare():
    with pytest.raises(ValueError, match="must be positive"):   # we *expect* this to fail
        tip_rate(0, 5)

In [ ]:
!cd testing_demo && pytest -q test_tips.py

A few things to notice:

 - `pytest.approx` for floats, since `0.1 + 0.2 == 0.3` is `False`.
 - `@pytest.mark.parametrize` runs one test over a table of cases, so seven tests ran from five functions.
 - `pytest.raises` checks that the code fails the way it should. Rejecting bad input is part of the behaviour, so it deserves a test too. (In the older `unittest` style this is `self.assertRaises`.)

## When a test fails

Now the agent "simplifies" `tip_rate` in a pull request. It looks reasonable:

In [ ]:
%%writefile testing_demo/tips.py
def tip_rate(fare, tip):
    """Tip as a fraction of the fare (simplified)."""
    return round(tip / fare, 2) if fare else 0.0

In [ ]:
!cd testing_demo && pytest -q --tb=short test_tips.py

pytest tells you exactly which tests failed and why. It even shows the values on both sides of the failed `assert`:

 - `test_thirds`: the result is now rounded to 2 decimal places, `0.33` rather than `0.3333`.
 - `test_rejects_zero_fare`: a zero fare now quietly returns `0.0` instead of raising, so pytest reports `DID NOT RAISE`. A zero fare is almost always a data error, and the old code made sure you heard about it.

Neither change is in the PR description. Writing a test first, then asking the agent to make it pass, is the tightest way to steer it: the test is an unambiguous acceptance criterion, and it keeps checking as the code changes underneath.

# Property-based testing: state a rule, let it hunt

Example tests only check the cases you thought of, and the cases you thought of are rarely the ones that break. A **property-based** test instead states a rule that should *always* hold, and the library generates lots of inputs trying to break it. In Python the library is **hypothesis** (`pip install hypothesis`).

In [ ]:
%%writefile testing_demo/test_normalise.py
from hypothesis import given, strategies as st

def normalise(xs):
    total = sum(xs)
    return [x / total for x in xs]          # should make the values sum to 1

@given(st.lists(st.integers(min_value=-10, max_value=10), min_size=1))
def test_sums_to_one(xs):
    assert abs(sum(normalise(xs)) - 1.0) < 1e-9

In [ ]:
!cd testing_demo && pytest -q --tb=short test_normalise.py

`@given` describes the *kind* of input (lists of small integers), and hypothesis tries a few hundred of them. It finds an input the author never pictured: a list that sums to zero. It also *shrinks* the failure to the simplest input that still breaks, so you get `xs=[0]` rather than some long random list.

For agent-written code this is ideal, because you do not have to imagine every edge case yourself.

## Fuzzing: testing the edges of your system

**Fuzzing** is the older, rougher relative of property-based testing: throw huge numbers of random or malformed inputs at a program and watch for crashes, hangs, or anything else it should not do. The "property" is usually just *it fails cleanly, with an error you meant to raise*.

It matters most at the **public-facing edges** of a system, anywhere input arrives that you do not control: a file upload, a web API, a command-line tool, a parser for someone else's file format. Inside your own pipeline you control the inputs, and example and property tests are usually enough. At the edge, users (and attackers) will eventually send exactly the input you never pictured, and a crash there can be a security problem rather than just a bug. That is the untrusted-input lesson from Module B again.

```python
@given(st.text())
def test_parse_duration_fails_cleanly(s):
    try:
        parse_duration(s)          # e.g. "1h30m", typed into a form by a user
    except ValueError:
        pass                       # a clear error is fine; any other exception is a bug
```

Large projects run fuzzers continuously. Coverage-guided fuzzers such as AFL and libFuzzer (Atheris for Python) watch which code each input reaches and mutate inputs to reach new code. Google's OSS-Fuzz runs these on open-source projects and has found tens of thousands of bugs. If your project has a dashboard or an API, a fuzz-style test of its input handling is cheap to write.

# Golden tests: test the output, not the function

Sometimes the code is too big or too tangled to test piece by piece, and you cannot easily say what the right answer *is*. But you can say "this run was right". A **golden** test saves that output once, as a reference file, and checks every later run against it. It tests the resulting *data*, so it does not care how the code inside works.

Say another team (or an agent) gave us `zone_report.py`, a few dozen lines of cleaning, joins and group-bys that turn the raw taxi trips into a per-zone summary table. We are not going to read it. We only call it.

In [ ]:
%%writefile testing_demo/zone_report.py
# Not shown in the lecture: this stands in for code you did not write and are not going to read.
import pandas as pd

def build_zone_report(trips, min_trips=20):
    df = trips[(trips["fare_amount"] >= 2.5) & (trips["trip_distance"] > 0)].copy()
    pickup = pd.to_datetime(df["tpep_pickup_datetime"])
    dropoff = pd.to_datetime(df["tpep_dropoff_datetime"])
    df["minutes"] = (dropoff - pickup).dt.total_seconds() / 60
    df = df[(df["minutes"] > 1) & (df["minutes"] < 180)]
    df["mph"] = df["trip_distance"] / (df["minutes"] / 60)
    df["night"] = pickup.loc[df.index].dt.hour < 6
    card = df[df["payment_type"] == 1]                      # cash tips are not recorded
    tip_pct = (100 * card["tip_amount"] / card["fare_amount"]).groupby(card["pulocationid"]).median()
    g = df.groupby("pulocationid")
    report = pd.DataFrame({
        "trips": g.size(),
        "median_fare": g["fare_amount"].median(),
        "mean_mph": g["mph"].mean(),
        "night_share": g["night"].mean(),
        "median_tip_pct": tip_pct,
    })
    report = report[report["trips"] >= min_trips].sort_values(["trips", "median_fare"], ascending=False)
    report.index.name = "zone"
    return report.round(3)

In [ ]:
%%writefile testing_demo/test_golden.py
from pathlib import Path
import pandas as pd
import pytest
from zone_report import build_zone_report

DATA = Path(__file__).parents[2] / "data" / "2019_Yellow_Taxi_Trip_Data.csv"
GOLDEN = Path(__file__).parent / "golden" / "zone_report.csv"

def test_zone_report_matches_golden():
    report = build_zone_report(pd.read_csv(DATA))
    if not GOLDEN.exists():                         # first run: record the reference
        GOLDEN.parent.mkdir(exist_ok=True)
        report.to_csv(GOLDEN)
        pytest.skip("recorded a new golden file")
    expected = pd.read_csv(GOLDEN, index_col="zone")
    pd.testing.assert_frame_equal(report, expected)

In [ ]:
!cd testing_demo && pytest -q -rs test_golden.py       # first run records the reference
!cd testing_demo && pytest -q test_golden.py           # later runs compare against it

In a real project you would check the golden file into git, after looking at it carefully once. The plugin `pytest-regressions` does this record-then-compare for you (`dataframe_regression.check(report)`).

Now the agent sends a pull request: "simplified the tip calculation in zone_report.py". We still do not read the code. We run the test.

In [ ]:
%%writefile testing_demo/zone_report.py
# Not shown in the lecture: the agent's "simplified" version. One line changed.
import pandas as pd

def build_zone_report(trips, min_trips=20):
    df = trips[(trips["fare_amount"] >= 2.5) & (trips["trip_distance"] > 0)].copy()
    pickup = pd.to_datetime(df["tpep_pickup_datetime"])
    dropoff = pd.to_datetime(df["tpep_dropoff_datetime"])
    df["minutes"] = (dropoff - pickup).dt.total_seconds() / 60
    df = df[(df["minutes"] > 1) & (df["minutes"] < 180)]
    df["mph"] = df["trip_distance"] / (df["minutes"] / 60)
    df["night"] = pickup.loc[df.index].dt.hour < 6
    card = df                                               # "simplified"
    tip_pct = (100 * card["tip_amount"] / card["fare_amount"]).groupby(card["pulocationid"]).median()
    g = df.groupby("pulocationid")
    report = pd.DataFrame({
        "trips": g.size(),
        "median_fare": g["fare_amount"].median(),
        "mean_mph": g["mph"].mean(),
        "night_share": g["night"].mean(),
        "median_tip_pct": tip_pct,
    })
    report = report[report["trips"] >= min_trips].sort_values(["trips", "median_fare"], ascending=False)
    report.index.name = "zone"
    return report.round(3)

In [ ]:
!cd testing_demo && pytest -q --tb=line test_golden.py

The golden test points straight at the column that changed: `median_tip_pct` is different in every zone (it now includes cash trips, whose tips are recorded as zero). We found that without reading a line of `zone_report.py`.

The catch is that a golden test cannot tell you whether the *old* output was right. It only tells you something changed. When a change is intended, you look at the difference, and if it is correct you re-record the golden file. That review step is the point.

## Scaling tests: make the performance budget a test

The performance budget from Module C becomes a test like any other. The `pytest-benchmark` plugin gives every test a `benchmark` fixture that runs a function several times and records the timings, which we can then assert on.

Two versions of the same aggregation, one of which rebuilds a subframe per zone:

In [ ]:
%%writefile testing_demo/aggregate.py
def agg_fast(df):
    return df.groupby("pulocationid")["tip_amount"].mean()

def agg_slow(df):
    return {z: df[df["pulocationid"] == z]["tip_amount"].mean()
            for z in df["pulocationid"].unique()}           # a subframe per zone: O(n*k)

In [ ]:
%%writefile testing_demo/test_scaling.py
from pathlib import Path
import pandas as pd
import pytest
from aggregate import agg_fast, agg_slow

DATA = Path(__file__).parents[2] / "data" / "2019_Yellow_Taxi_Trip_Data.csv"
BUDGET_S = 0.1

@pytest.fixture(scope="module")
def big():
    # a realistic size (~1 million rows), built once and shared by every test in the file
    return pd.concat([pd.read_csv(DATA)] * 100, ignore_index=True)

@pytest.mark.parametrize("agg", [agg_fast, agg_slow])
def test_within_budget(benchmark, big, agg):
    benchmark.pedantic(agg, args=(big,), rounds=3)
    mean = benchmark.stats.stats.mean
    assert mean < BUDGET_S, f"{mean * 1000:.0f} ms is over the {BUDGET_S * 1000:.0f} ms budget"

In [ ]:
!cd testing_demo && pytest -q --tb=line --benchmark-columns=mean,rounds test_scaling.py

Both versions return the same answer, so every correctness test passes. The scaling test catches the slow one, at a realistic size, before it reaches the full dataset. A **fixture** (`big`) is how pytest shares set-up between tests: any test that names `big` as an argument gets the same million-row frame.

Other timing tools in the pytest world:

 - `pytest --durations=5` lists the five slowest tests in any run, with no plugin needed.
 - `pytest-timeout` (`@pytest.mark.timeout(10)`) fails a test that hangs or runs too long.
 - `pytest-benchmark` can save a baseline (`--benchmark-autosave`) and fail if a later run gets slower than it (`--benchmark-compare-fail=mean:20%`), which catches a slow-down without picking an absolute budget. That is more robust, since the budget depends on the machine.

# Validate the data, not just the code

Code can be correct and still produce nonsense if the *data* is wrong. Checks on a DataFrame's shape (its columns, types, ranges, row counts and keys) are the automated version of the silent-failure catalogue from Module A. Here is the many-to-many join blow-up from Module C again:

In [ ]:
import pandas as pd

left = pd.DataFrame({"zone": [1, 1, 2], "trips": [10, 20, 30]})
right = pd.DataFrame({"zone": [1, 1, 2], "name": ["A", "B", "C"]})
merged = left.merge(right, on="zone")           # duplicate keys: 3 + 3 rows -> 5
print(len(merged), "rows")

Remember that pandas can check some of this for you. `merge` takes a `validate=` argument saying what kind of join you *expect*, and raises an error if the keys do not fit:

In [ ]:
try:
    left.merge(right, on="zone", validate="many_to_one")    # each left row should match at most one name
except pd.errors.MergeError as e:
    print("MergeError:", str(e).splitlines()[0])

`validate=` only checks that the keys are unique where you said they would be. It does **not** check keys that fail to match, missing values, or whether the numbers make sense. This merge passes validation and quietly gives zone 2 no name:

In [ ]:
names = pd.DataFrame({"zone": [1, 3], "name": ["A", "C"]})
trips = pd.DataFrame({"zone": [1, 2], "trips": [10, -30]})
merged = trips.merge(names, on="zone", how="left", validate="one_to_one", indicator=True)
print(merged)

`indicator=True` adds a `_merge` column that shows which rows did not match (`left_only`). For the rest, a small check function covers what pandas does not:

In [ ]:
def check(df):
    problems = []
    if (df["_merge"] != "both").any():
        problems.append(f"{(df['_merge'] != 'both').sum()} row(s) with no match")
    if df["name"].isna().any():
        problems.append("missing names")
    if (df["trips"] <= 0).any():
        problems.append("trip counts must be positive")
    return problems

print(check(merged))

`pandera` lets you write checks like these as a declarative schema (column types, allowed ranges, uniqueness) and attach it to a pipeline stage, so every stage validates its own output. Either way, a cheap check on the shape of the data stops a quiet blow-up from flowing downstream into your results.

# Reviewing an agent's pull request

Put it together. Here is a pull request from an agent. The task was "return the zones with the highest average tip, and make it fast." It comes with tests, and says they all pass.

In [ ]:
%%writefile testing_demo/pr_tips.py
def top_tip_zones(df):
    zones = df["pulocationid"].unique()
    out = []
    for z in zones:
        trips = df[df["pulocationid"] == z]            # scans the whole frame, per zone
        out.append({"zone": z, "avg_tip": trips["tip_amount"].mean()})
    return sorted(out, key=lambda r: r["avg_tip"])[:5]

In [ ]:
%%writefile testing_demo/test_pr_tips.py
import pandas as pd
from pr_tips import top_tip_zones

def make_trips():
    return pd.DataFrame({"pulocationid": [1, 1, 2, 2, 3],
                         "tip_amount":   [2.0, 2.0, 2.0, 2.0, 2.0]})

def test_returns_at_most_five():
    assert len(top_tip_zones(make_trips())) <= 5

def test_rows_have_zone_and_avg_tip():
    for row in top_tip_zones(make_trips()):
        assert set(row) == {"zone", "avg_tip"}

def test_finds_the_zones():
    zones = {row["zone"] for row in top_tip_zones(make_trips())}
    assert zones == {1, 2, 3}

In [ ]:
!cd testing_demo && pytest -q test_pr_tips.py

### Question 2

The PR claims to return the highest-tipping zones, quickly, with all tests passing, and the tests really do pass. Which problem is real? (There is more than one; pick the one you would raise first.)

 - A. It sorts ascending, so it returns the **lowest** five, not the highest
 - B. The per-zone loop scans the whole frame each time, so it is O(n*k)
 - C. It averages in cash trips, which record a tip of 0, dragging the numbers down
 - D. The tests are too loose to catch any of these

All four are real, which is the point of a review. **A** is a correctness bug (it returns the wrong end of the list). **B** is the Module C scaling problem (a `groupby` is O(n)). **C** is the Module A silent-data bug (cash tips are recorded as 0). **D** is the Module B trap: the tests pass, but look at what they test.

 - Every zone has the **same** average tip, so the order is never checked, and bug A cannot show up.
 - There are only three zones, so the "top five" cut is never exercised.
 - There is no `payment_type` column at all, so cash trips (bug C) cannot appear.
 - Nothing checks speed (bug B).

These tests would pass for almost any function that returns a list of dicts. That is what "all tests pass" can hide.

## Writing the tests the PR should have had

Two tests, each built so the right answer is obvious and the bug has somewhere to show up: distinct averages so the order matters, more than five zones, and a cash trip in the data.

In [ ]:
%%writefile testing_demo/test_review.py
import pandas as pd
from pr_tips import top_tip_zones

def test_highest_first():
    # seven zones with distinct average tips: zone z tips z dollars
    trips = pd.DataFrame({"pulocationid": list(range(1, 8)),
                          "tip_amount": [float(z) for z in range(1, 8)],
                          "payment_type": 1})
    assert [int(r["zone"]) for r in top_tip_zones(trips)] == [7, 6, 5, 4, 3]

def test_ignores_cash_trips():
    # zone 1: card tips of $3, plus a cash trip recorded as $0
    trips = pd.DataFrame({"pulocationid": [1, 1, 1],
                          "tip_amount": [3.0, 3.0, 0.0],
                          "payment_type": [1, 1, 2]})
    assert float(top_tip_zones(trips)[0]["avg_tip"]) == 3.0

In [ ]:
!cd testing_demo && pytest -q --tb=line test_review.py

Both fail on the PR, which is what we want: now the bugs are written down as checks, and "make these pass" is a precise instruction to send back to the agent. Add the scaling test from earlier and you have caught A, B and C, without having to spot any of them by reading the code.

## CI: run the checks on every change

The last piece is making the checks run themselves. **Continuous integration** (GitHub Actions) runs `pytest` on every push, so a change that breaks a test is caught immediately, by the machine, not weeks later by a user. It is already wired into the project template, so your M3 repository runs its tests on every commit.

# Recap

 - Test the **behaviour**, not the implementation, because the agent may rewrite the implementation.
 - **pytest**: `test_` functions that assert, with `approx`, `parametrize`, `raises` and fixtures. Read the failure report; it shows you the values.
 - **Property-based** tests (`hypothesis`) state a rule and hunt for a counterexample; **fuzzing** does the same at the public edges of a system.
 - **Golden** tests check the output data of code you do not want to read; **scaling** tests (`pytest-benchmark`) make the performance budget a check.
 - **Validate the data** too: `merge(validate=..., indicator=True)` covers the keys, a schema check (`pandera`) covers the rest.
 - In a green pull request, read the **tests** first, and ask what they would fail to catch. **CI** runs it all on every push.

**Next time (S24): shaping the build** - deciding what to build, and what "done" means.